The following example shows how to log data for training. The log data will be saved into data/training folder

In [2]:
# %matplotlib tk
import matplotlib.pyplot as plt
import sys
import os
import numpy as np

# Get the current directory of the notebook
notebook_dir = os.getcwd()

# Add the parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

import manager
import drone.trajectory
import drone.disturbance_model

plt.rcParams['figure.figsize'] = [15, 9]

wall_distance = 0.5
# wall_distance = 100.0
wind_speed_x = -0.0
wind_speed_y = 0.0
wind_speed_z = -5.0

sim_test = manager.Manager()
model = drone.disturbance_model.WindEffectNearWall(wall_origin=np.array([-wall_distance, 0, 0]), 
                                                    u_free = np.array([wind_speed_x, wind_speed_y, wind_speed_z]))
# model = drone.disturbance_model.AggregatedDisturbance(wall_origin=np.array([-wall_distance, 0, 0]), 
#                                                     u_free = np.array([wind_speed_x, wind_speed_y, wind_speed_z]))
# model = drone.disturbance_model.Free()    # debug only
# trajectory=drone.trajectory.RandomWaypointsInConstrainedSpace(200, False)
trajectory = drone.trajectory.Figure8(5)
# trajectory= drone.trajectory.CircleYZ()
# trajectory= drone.trajectory.StraightNegativeX()
# trajectory = drone.trajectory.Hover()
sim_test.set_up(disturbance_model=model, trajectory=trajectory)

sim_test.run(6.0)   # auto-saves ALL logged fields to data/runs/<timestamp>.csv (git-ignored)
# sim_test.save_result_as_csv('test')
# sim_test.plot()   # live path via the Manager still works

# --- Decoupled plotting: reload the latest full dump and plot without the Manager/Engine ---
# (also works from a fresh kernel -- just run this block on its own)
import sim_logger, plotter
output = sim_logger.Logger.load_full_log()      # latest dump in data/runs/ (or pass a file path)
result = plotter.Plotter.from_output(output)    # t_span reconstructed from the data (dt=0.01)
result.make_plots(output)
plt.show()

[PropellerLookupTable] Reading data from ..\inflow_model\lookup_table\p600_single_rotor_after_bemt_fit.yaml
is_sinusoidal_wind:  False
Map data read from: ..\data\map\figure_8.pkl
Map data read from: ..\data\map\figure_8.pkl
Map data read from: ..\data\map\figure_8.pkl
Map data read from: ..\data\map\figure_8.pkl
Map data read from: ..\data\map\figure_8.pkl
-----------external simulator init-----------
wind_speed:  [-0.  0. -5.]
init position:  [1.61908872 0.         0.        ]
DroneController: using inflow model:  True
DroneController: closed-loop disturbance estimator:  SEQUENCE
phi net input output fields:
input {'position': [0, 1, 2], 'q': [0, 1, 2, 3], 'v': [0, 1, 2], 'sensed_omega': [0, 1, 2], 'f_ctrl_input': [2], 'torque_ctrl_input': [0, 1, 2], 'rotor_0_rotation_spd': None, 'rotor_1_rotation_spd': None, 'rotor_2_rotation_spd': None, 'rotor_3_rotation_spd': None}
label {'f_disturb_sensed_raw': [0, 1, 2]}


KeyError: 'dim_of_input'

To compare the desired pose and actual pose, use the following command.

In [ ]:
%matplotlib inline
t = 0.6
result.plot_3d_trace(output, t)
plt.show()

import inflow_model.propeller_lookup_table as propeller_lookup_table
from inflow_model.blade_params import P600_Blade
from learning.bemt_traditional_fit.manager import FittingManager

lt = propeller_lookup_table.PropellerLookupTable.Reader("p600_single_rotor_quadratic")
is_ccw = [False, True, False, True]

fitted_params = [19.788, 4.183, 0.011, np.radians(5.989)]
bet_manager = FittingManager.for_single_rotor(P600_Blade(), is_ccw_rotor0=False, datasets=[])
bet_model = bet_manager.model
bet_model.blade.cl_1, bet_model.blade.cl_2, bet_model.blade.cd, bet_model.blade.alpha_0 = fitted_params
bet_model.bet_instance.refresh_blade()
bet_model.adjust_resolution(is_fine_tune=True)
result.print_wind_at_time(output, t, lt, is_ccw)
result.print_rotor_forces_at_time(output, t=t,
                                            lookup_table=lt, is_ccw_blade=is_ccw,
                                            bet_model=bet_model)

Pop out a specific figure to zoom in.

In [ ]:
%matplotlib tk
result.plot_disturbance_force(output)
result.plot_rotor(output)
result.plot_3d_trace(output)
result.plot_task_trajectory(output)
result.plot_force_and_torque(output)

# sim_test.plot()

Summarize rms metrics.

In [ ]:
rms_x = np.sqrt(np.mean(np.square(output["e_x"][:, 0])))
rms_y = np.sqrt(np.mean(np.square(output["e_x"][:, 1])))
rms_z = np.sqrt(np.mean(np.square(output["e_x"][:, 2])))
rms_norm = np.sqrt(np.mean(output["e_x"][:, 0]**2 + output["e_x"][:, 1]**2 + output["e_x"][:, 2]**2))
print(f"RMS position error: \nx: {rms_x}, \ny: {rms_y}, \nz: {rms_z}, \nnorm: {rms_norm}")